In [0]:
from common_scripts.common_functions import setup_adls_access

setup_adls_access(spark, dbutils)

In [0]:
bronze_path = "abfss://financial-data-project@financialdatalakestorage.dfs.core.windows.net/bronze/crypto/"
df_raw = spark.read.json(bronze_path)
display(df_raw)

In [0]:
from pyspark.sql.functions import col

df_clean = df_raw.select(
    col("id"),
    col("symbol"),
    col("name"),
    col("current_price").cast("double"),
    col("market_cap").cast("long"),
    col("total_volume").cast("long"),
    col("high_24h").cast("double"),
    col("low_24h").cast("double"),
    col("price_change_percentage_24h").cast("double"),
    col("ingestion_time"),
    col("ingestion_date").alias("data_load_date")
)

display(df_clean)

In [0]:
df_clean = df_clean.dropDuplicates(["id", "ingestion_time"])

In [0]:
silver_path = "abfss://financial-data-project@financialdatalakestorage.dfs.core.windows.net/silver/crypto"

df_clean.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("data_load_date") \
    .save(silver_path)

In [0]:
display(dbutils.fs.ls(silver_path))

In [0]:
%sql
DROP TABLE IF EXISTS adb_financial_datalake.financial_project_db.silver_crypto;
CREATE TABLE adb_financial_datalake.financial_project_db.silver_crypto
USING DELTA
LOCATION 'abfss://financial-data-project@financialdatalakestorage.dfs.core.windows.net/silver/crypto';

In [0]:
%sql
SELECT * FROM adb_financial_datalake.financial_project_db.silver_crypto;
